# Module 3 — Scène Three.js et coordonnées

**Objectif :** Relier caméra, coordonnées normalisées, objet sélectionné et alternative HTML.

Activité de **30 minutes**, incluse dans les **3 heures solo** : **2 h lecture guidée → 30 min Colab → 30 min bilan**. Le webinaire de **1 heure** a lieu après la préparation autonome. Formation complète : **10 h accompagnées + 30 h solo = 40 h**.

CPU gratuit, sans GPU, API payante ou IA Colab obligatoire. Rien à installer. Les sorties JS sont exécutées par votre navigateur. Three.js, lorsque nécessaire, est chargé depuis un CDN à version fixée; l’alternative HTML reste disponible si le réseau ou WebGL est indisponible.

Faites une copie personnelle dans Drive. Vos productions sont conservables et réutilisables. Le partage du notebook peut partager son code, ses textes et ses sorties. Ni votre journal ni vos fichiers ne sont envoyés dans Sanity automatiquement.

**Trois petites étapes :** prédire (5 min), changer/observer (15 min), expliquer/exporter (10 min). Vous pouvez demander une réponse à votre assistant; indiquez l’aide reçue et vérifiez son application.

In [ ]:
import base64, hashlib, html, io, json, math, re, time, zipfile
from pathlib import Path
from IPython.display import HTML, display

# CPU standard; rien à installer. Les sorties HTML s'exécutent dans votre navigateur.
MODULE_ID = 3
MISSION_ID = 'module-' + str(MODULE_ID)
reflection = {
    'prediction': '', 'observations': [], 'explanation': '',
    'assistance': '', 'limitations': '', 'openQuestion': ''
}
parameters = {}
print('Préparation prête. Votre notebook appartient à votre compte Google; le partager peut partager code et sorties.')

## 1. Prédire — 5 min
En reculant la caméra, les positions enregistrées changent-elles ou seulement leur projection ?

Notez votre prédiction avant de voir l’effet. Une prédiction imparfaite est une trace utile.

In [ ]:
reflection['prediction'] = ''  # Votre hypothèse AVANT l’essai
parameters = {}  # Notez la valeur initiale, la valeur testée et les conditions conservées.

## 2. Modifier et observer — 15 min
Comparez cameraDistance 5 et 7. Gardez les positions identiques. Cliquez ensuite une sphère puis un bouton HTML.

Le fichier est une chaîne Python contenant du JavaScript/Astro réel. L’aperçu et l’export utilisent **la même** variable `student_files`. Conservez les exports nommés et les signatures; un changement hors contrat doit être examiné plutôt que remplacé silencieusement.

In [ ]:
student_files = {}
student_files['scene-controller.js'] = r'''export const sceneOptions = { cameraDistance: 5 }; // STUDENT: compare 5 and 7, with the same objects.
/** THREE is the exact host dependency, never a second bundled library. No owned RAF. */
export function createSceneController({ container, THREE, items, onSelect, pointerTarget, options = sceneOptions }) {
  if (!(options.cameraDistance >= 1 && options.cameraDistance <= 10)) throw new Error('Camera distance outside the learning limits.');
  let renderer; const lifetime = new AbortController(); const geometries = []; const materials = [];
  try { renderer = new THREE.WebGLRenderer({ antialias: true, alpha: true }); }
  catch { return { available: false, render() {}, setPosition() {}, setScale() {}, setParticles() {}, dispose() {} }; }
  renderer.setPixelRatio(Math.min(globalThis.devicePixelRatio || 1, 1.5)); renderer.domElement.setAttribute('aria-hidden', 'true'); container.append(renderer.domElement);
  Object.assign(renderer.domElement.style, { display: 'block', width: '100%', height: '100%' });
  const scene = new THREE.Scene(); const camera = new THREE.PerspectiveCamera(45, 1, 0.1, 50);
  camera.position.z = options.cameraDistance;
  scene.add(new THREE.HemisphereLight(0xffffff, 0x172037, 2));
  const objects = new Map();
  for (const [index, item] of items.entries()) {
    const geometry = new THREE.SphereGeometry(0.22, 16, 12);
    const material = new THREE.MeshStandardMaterial({ color: [0x5cd6e8, 0xa98de6, 0xffa234][index % 3] });
    geometries.push(geometry); materials.push(material);
    const object = new THREE.Mesh(geometry, material); object.userData.id = item.id;
    object.position.set((item.position.x - 0.5) * 4, (0.5 - item.position.y) * 3, 0); scene.add(object); objects.set(item.id, object);
  }
  const pointGeometry = new THREE.BufferGeometry(); const pointMaterial = new THREE.PointsMaterial({ color: 0xffa234, size: 0.025 });
  geometries.push(pointGeometry); materials.push(pointMaterial); const points = new THREE.Points(pointGeometry, pointMaterial); points.frustumCulled = false; scene.add(points);
  let pointArray = new Float32Array(0);
  const resize = () => { const rect = container.getBoundingClientRect(); camera.aspect = Math.max(1, rect.width) / Math.max(1, rect.height); camera.updateProjectionMatrix(); renderer.setSize(Math.max(1, rect.width), Math.max(1, rect.height), false); renderer.render(scene, camera); };
  const observer = new ResizeObserver(resize); observer.observe(container); resize();
  const ray = new THREE.Raycaster(); let start;
  // The host can share its pointer-capture surface; projection still uses the canvas.
  const inputTarget = pointerTarget ?? renderer.domElement;
  inputTarget.addEventListener('pointerdown', (event) => {
    if (event.target !== renderer.domElement && !event.composedPath().includes(renderer.domElement)) { start = undefined; return; }
    start = { x: event.clientX, y: event.clientY };
  }, { signal: lifetime.signal });
  inputTarget.addEventListener('pointercancel', () => { start = undefined; }, { signal: lifetime.signal });
  inputTarget.addEventListener('pointerup', (event) => {
    const gesture = start; start = undefined;
    if (!gesture || Math.hypot(event.clientX - gesture.x, event.clientY - gesture.y) > 6) return;
    const rect = renderer.domElement.getBoundingClientRect();
    ray.setFromCamera(new THREE.Vector2((event.clientX - rect.left) / rect.width * 2 - 1, -((event.clientY - rect.top) / rect.height) * 2 + 1), camera);
    const hit = ray.intersectObjects([...objects.values()])[0]; if (hit) onSelect(hit.object.userData.id);
  }, { signal: lifetime.signal });
  return { available: true,
    setPosition(id, position) { const object = objects.get(id); if (object) object.position.set((position.x - 0.5) * 4, (0.5 - position.y) * 3, 0); },
    setScale(id, value) { objects.get(id)?.scale.setScalar(value); },
    setParticles(particles) {
      if (pointArray.length !== particles.length * 3) { pointArray = new Float32Array(particles.length * 3); pointGeometry.setAttribute('position', new THREE.BufferAttribute(pointArray, 3)); }
      particles.forEach((particle, index) => { pointArray[index * 3] = (particle.x - 0.5) * 4; pointArray[index * 3 + 1] = (0.5 - particle.y) * 3; pointArray[index * 3 + 2] = -0.3; });
      if (pointArray.length) pointGeometry.attributes.position.needsUpdate = true;
      points.visible = Boolean(particles.length);
    }, render() { renderer.render(scene, camera); },
    dispose() { lifetime.abort(); observer.disconnect(); for (const value of geometries) value.dispose(); for (const value of materials) value.dispose(); renderer.dispose(); renderer.domElement.remove(); }
  };
}
'''
# Modifiez le texte ci-dessus, puis réexécutez cette cellule et l’aperçu.

In [ ]:
PREVIEW_TEMPLATE = '<!doctype html><html lang="fr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width">\n<style>body{font:16px system-ui;color:#172033;background:#f5f7fb;padding:1rem}button{margin:.25rem;padding:.5rem}.stage{height:270px;position:relative;background:#11152a;color:white;overflow:hidden;touch-action:none;outline-offset:2px}.marker{position:absolute;background:#ffa234;width:24px;height:24px;border-radius:50%;transform:translate(-50%,-50%);pointer-events:none}canvas{display:block;width:100%;height:100%}pre{white-space:pre-wrap;overflow-wrap:anywhere}.error{color:#a31515}label{display:inline-block;margin:.5rem}li{margin:.5rem}</style></head><body>\n<p>Une variable à la fois. Gardez votre prédiction visible dans le notebook.</p>\n<label><input id="static" type="checkbox"> Figer l’image et le temps</label><button id="restart">Recharger l’aperçu</button>\n<div id="stage" class="stage" tabindex="0" aria-label="Zone d’expérience : utilisez aussi les flèches"><span id="marker" class="marker"></span></div>\n<section id="cards"><ul data-learning-cards></ul><p data-learning-description aria-live="polite"></p></section>\n<p id="info" role="status"></p><pre id="trace"></pre><p id="error" class="error" role="alert"></p>\n<script type="module">\nconst studentFiles = __STUDENT_FILES__;\nconst lifetimes = [];\nconst urls = Object.fromEntries(Object.entries(studentFiles).filter(([name]) => name.endsWith(\'.js\')).map(([name,text]) => [name, URL.createObjectURL(new Blob([text], {type:\'text/javascript\'}))]));\nconst stage=document.getElementById(\'stage\'), marker=document.getElementById(\'marker\'), info=document.getElementById(\'info\'), trace=document.getElementById(\'trace\');\nconst staticControl=document.getElementById(\'static\'); let isStatic=matchMedia(\'(prefers-reduced-motion: reduce)\').matches, last=0, frame=0;\nstaticControl.checked=isStatic;\nconst show=(value)=>{trace.textContent=JSON.stringify(value,null,2)};\nconst point=(event)=>{const r=stage.getBoundingClientRect();return {x:Math.max(0,Math.min(1,(event.clientX-r.left)/r.width)),y:Math.max(0,Math.min(1,(event.clientY-r.top)/r.height))}};\nconst drawPoint=(p)=>{marker.style.left=(p.x*100)+\'%\';marker.style.top=(p.y*100)+\'%\'};\nlet step=()=>{}, onStatic=()=>{};\nfunction loop(now){if(isStatic)return;const dt=last?Math.min((now-last)/1000,.05):0;last=now;step(dt);frame=requestAnimationFrame(loop)}\nstaticControl.addEventListener(\'change\',()=>{isStatic=staticControl.checked;onStatic(isStatic);cancelAnimationFrame(frame);last=0;if(!isStatic)frame=requestAnimationFrame(loop)});\ndocument.getElementById(\'restart\').addEventListener(\'click\',()=>location.reload());\nwindow.addEventListener(\'pagehide\',()=>{cancelAnimationFrame(frame);lifetimes.forEach(f=>f());Object.values(urls).forEach(URL.revokeObjectURL)});\nconst items=[{id:\'element-1\',title:\'Geste\',content:\'Une position enregistrée.\',position:{x:.5,y:.5}},{id:\'element-2\',title:\'Observation\',content:\'Un résultat à examiner.\',position:{x:.25,y:.3}},{id:\'element-3\',title:\'Transfert\',content:\'Une nouvelle situation.\',position:{x:.75,y:.3}}];\ntry {\nmarker.hidden=true;\ntry { const THREE=await import(\'https://cdn.jsdelivr.net/npm/three@0.181.2/build/three.module.js\');\n const {createSceneController}=await import(urls[\'scene-controller.js\']);\n const scene=createSceneController({container:stage,THREE,items,onSelect:id=>show({selected:id,source:\'3D\',position:items.find(i=>i.id===id).position})});lifetimes.push(()=>scene.dispose());step=()=>scene.render();\n info.textContent=scene.available?\'Three.js 0.181.2, dans votre navigateur. Caméra et position sont distinctes.\':\'WebGL indisponible : utilisez les boutons HTML.\';\n} catch(error) { info.textContent=\'CDN ou WebGL indisponible; alternative HTML conservée : \'+error.message; }\nconst list=document.querySelector(\'[data-learning-cards]\');items.forEach(item=>{const li=document.createElement(\'li\'),button=document.createElement(\'button\');button.textContent=item.title;button.onclick=()=>show({selected:item.id,source:\'HTML\',position:item.position});li.append(button);list.append(li)});\nonStatic(isStatic);step(0);if(!isStatic)frame=requestAnimationFrame(loop);\n} catch(error) { document.getElementById(\'error\').textContent=\'Erreur de l’essai : \'+error.message+\' — corrigez le fichier ou reprenez sa version précédente.\'; }\n</script></body></html>'
PREVIEW_CODE_TEXT = 'if not reflection[\'prediction\'].strip():\n    raise ValueError(\'Écrivez votre prédiction avant l’essai, puis exécutez cette cellule à nouveau.\')\nfor name, source in student_files.items():\n    if not isinstance(source, str) or len(source.encode(\'utf-8\')) > 200_000:\n        raise ValueError(\'Fichier élève trop gros ou non textuel.\')\npreview_document = PREVIEW_TEMPLATE.replace(\'__STUDENT_FILES__\', json.dumps(student_files, ensure_ascii=False).replace(\'<\', r\'\\u003c\'))\nencoded = base64.b64encode(preview_document.encode(\'utf-8\')).decode(\'ascii\')\ndisplay(HTML(\'<iframe title="Expérience frontend du module" sandbox="allow-scripts allow-same-origin" \'\n             \'style="width:100%;height:620px;border:1px solid #8796b4" src="data:text/html;base64,\' + encoded + \'"></iframe>\'))\nprint(\'Aperçu navigateur du code dans student_files. Ce résultat n’est pas une compilation Astro ni un appel WebMCP certifié.\')\n'
INTEGRATION_TEXT = '# Raccordement — module 3\n\n- `src/learning/module-3/scene-controller.js`\n\nCes fichiers proviennent de votre variable student_files; aucun corrigé ne remplace le code à l’export.\nConservez identifiants, coordonnées normalisées, dt en secondes et fonctions de nettoyage.\nL’aperçu n’est pas une compilation Astro, un appel WebMCP réel ni une mesure de performance Three.js.\nLa remise reste sélectionnée par vous. Sanity ne reçoit rien de ce notebook.\nLe notebook.ipynb exporté conserve le code et le bilan pour reprise, sans prétendre capturer l’exécution indépendante.\n\nTransfert : Conservez une fiche HTML lorsqu’une carte 3D est indisponible.\n'

In [ ]:
if not reflection['prediction'].strip():
    raise ValueError('Écrivez votre prédiction avant l’essai, puis exécutez cette cellule à nouveau.')
for name, source in student_files.items():
    if not isinstance(source, str) or len(source.encode('utf-8')) > 200_000:
        raise ValueError('Fichier élève trop gros ou non textuel.')
preview_document = PREVIEW_TEMPLATE.replace('__STUDENT_FILES__', json.dumps(student_files, ensure_ascii=False).replace('<', r'\u003c'))
encoded = base64.b64encode(preview_document.encode('utf-8')).decode('ascii')
display(HTML('<iframe title="Expérience frontend du module" sandbox="allow-scripts allow-same-origin" '
             'style="width:100%;height:620px;border:1px solid #8796b4" src="data:text/html;base64,' + encoded + '"></iframe>'))
print('Aperçu navigateur du code dans student_files. Ce résultat n’est pas une compilation Astro ni un appel WebMCP certifié.')

## 3. Expliquer et exporter — 10 min
Comparez prédiction et observation. Expliquez une ligne ou une décision avec vos mots. Indiquez votre aide reçue, une limite et une prochaine question.

**Transfert proposé :** Conservez une fiche HTML lorsqu’une carte 3D est indisponible.

Le bilan de 30 minutes qui suit cette activité sert à choisir ce que vous remettez à l’enseignant et à préparer sa revue. Un ZIP valide ou une empreinte ne prouve pas une maîtrise.

In [ ]:
reflection['observations'] = []
reflection['explanation'] = ''
reflection['assistance'] = ''  # Écrivez aussi « aucune » si pertinent.
reflection['limitations'] = ''
reflection['openQuestion'] = ''

In [ ]:
# Faites votre bilan avant de rendre. Les observations sont déclarées par vous.
required = ['prediction', 'explanation', 'assistance', 'limitations']
if any(not isinstance(reflection.get(key), str) or not reflection[key].strip() for key in required):
    raise ValueError('Complétez prédiction, explication, aide reçue (ou aucune) et limite avant le rendu.')
if not reflection.get('observations'):
    raise ValueError('Conservez au moins une observation sélectionnée.')
if any(len(str(value)) > 12000 for value in reflection.values()):
    raise ValueError('Le bilan est trop long; sélectionnez les éléments utiles, pas toute la conversation.')
attempt_id = 'module-' + str(MODULE_ID) + '-' + str(time.time_ns())
artifacts = []
export_files = {}
for name, source in student_files.items():
    if not re.fullmatch(r'[A-Za-z0-9_.-]+\.(js|astro)', name):
        raise ValueError('Chemin frontend non autorisé : ' + name)
    raw = source.encode('utf-8')
    if len(raw) > 200_000:
        raise ValueError('Brique trop volumineuse.')
    path = 'frontend/' + name
    export_files[path] = raw
    artifacts.append({'id': 'module-' + str(MODULE_ID) + '-' + name, 'path': path, 'content': source,
                      'mediaType': 'text/plain', 'sha256': hashlib.sha256(raw).hexdigest()})
result = {'schemaVersion': 'orbit-learning-colab-v1', 'missionId': MISSION_ID, 'moduleId': MODULE_ID,
          'attemptId': attempt_id, 'parameters': parameters, **reflection,
          'status': 'external-declared', 'artifacts': artifacts,
          'verification': {'sourceCaptured': True, 'independentExecutionVerified': False,
                           'humanApproval': False, 'masteryCertified': False}}
export_files['orbit-learning-result.json'] = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False).encode('utf-8')
export_files['INTEGRATION.md'] = INTEGRATION_TEXT.encode('utf-8')

# Copie rejouable de votre code effectif, non une capture certifiée des sorties du navigateur.
def cell(kind, source):
    value = {'cell_type': kind, 'metadata': {}, 'source': source.splitlines(True)}
    if kind == 'code': value.update(outputs=[], execution_count=None)
    return value
record = {'nbformat': 4, 'nbformat_minor': 5,
          'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'},
                       'orbit': {'kind': 'source-capture-replay', 'notIndependentExecutionProof': True}},
          'cells': [cell('markdown', '# Mon rendu — module ' + str(MODULE_ID) + '\nCode capturé depuis student_files; observations déclarées. Les sorties navigateur ne sont pas archivées automatiquement.'),
                    cell('code', 'from IPython.display import HTML, display\nimport base64, json\nstudent_files = ' + repr(student_files) + '\nreflection = ' + repr(reflection) + '\nparameters = ' + repr(parameters)),
                    cell('code', 'PREVIEW_TEMPLATE = ' + repr(PREVIEW_TEMPLATE) + '\n' + PREVIEW_CODE_TEXT),
                    cell('markdown', 'Revue : reformuler une décision, refaire un essai et tenter un transfert.') ]}
export_files['notebook.ipynb'] = json.dumps(record, ensure_ascii=False, indent=2).encode('utf-8')
manifest = {'schemaVersion': 'orbit-learning-artifact-bundle-v1', 'missionId': MISSION_ID,
            'attemptId': attempt_id, 'status': 'external-declared',
            'files': [{'path': path, 'sha256': hashlib.sha256(raw).hexdigest()} for path, raw in sorted(export_files.items())]}
export_files['manifest.json'] = json.dumps(manifest, ensure_ascii=False, indent=2).encode('utf-8')
archive_buffer = io.BytesIO()
# ZIP_STORED makes the bounded browser import possible without an extra decompression dependency.
with zipfile.ZipFile(archive_buffer, 'w', compression=zipfile.ZIP_STORED) as archive:
    for path, raw in sorted(export_files.items()):
        archive.writestr(path, raw)
filename = 'orbit-module-' + str(MODULE_ID) + '-' + attempt_id + '.zip'
Path(filename).write_bytes(archive_buffer.getvalue())
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print('Fichier prêt dans le répertoire du notebook :', filename)
print(json.dumps({'state': 'EXPORTED_DECLARED_WORK', 'moduleId': MODULE_ID,
                  'files': len(export_files), 'sha256': hashlib.sha256(archive_buffer.getvalue()).hexdigest()}))